### State of the Climate Report for 2025

This analysis is for the chlorophyll-a part of the Southern Ocean section of the report.

Lead Author of the Southern Ocean section: Sandy Thomalla (Southern Ocean Carbon and Climate Observatory)

Data: ESA Ocean Colour Climate Change Initiative (OC-CCI) v6.0, Level-3 merged 8-day composites, 4 km, accessed via THREDDS OPeNDAP: https://www.oceancolour.org/thredds/dodsC/CCI_ALL-v6.0-8DAY

Plotting style, sector definitions and the standard-error method follow the heat-flux analysis for the same section: https://github.com/marcelduplessis/bams-soc-report-so-heat-fluxes

**Seasonal year.** Chlorophyll is analysed on a July–June year so the austral growing season is not split. The year label is the year the season ends, so **2025 = 1 July 2024 – 30 June 2025** (Jul–Dec from 2024, Jan–Jun from 2025).

# Load packages

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from functions import preprocess as pp
from functions.analysis import (cell_area, sector_means, monthly_in_season_order, SEASON_MONTHS)
from functions.map_plotting import (load_orsi_front, southern_ocean_anomaly_map, annual_cycle_plot)

# Settings

In [ ]:
anomyear = 2025                # season Jul 2024 - Jun 2025
climatology = [1999, 2020]     # seasons; OC-CCI begins Sep 1997, so 1999 is the first complete Jul-Jun season
first_season = 1999
variable = 'chlor_a'

# data are too large for the repo, so they live outside it
DATA = '../DATA'
dir_8day    = f'{DATA}/8day'
dir_annual  = f'{DATA}/annual'
dir_monthly = f'{DATA}/monthly'
dir_output  = f'{DATA}/output'
dir_fronts  = f'{DATA}/fronts'
dir_figures = '../FIGURES'

# map: average 6x6 4-km cells to 0.25 deg before contouring (same grid as ERA5 in the
# heat-flux map, and much faster to draw). Set to 1 to contour the full 4-km field.
map_coarsen = 6
map_levels = np.arange(-0.4, 0.45, 0.05)

dataset, frequency, resolution = 'oc-cci', '8day', '4km'
climstr = f'{climatology[0]}-{climatology[1]}'

# Downloading OC-CCI data

## Download 8 day data

This code accesses the ESA Ocean Colour Climate Change Initiative (OC-CCI) Level-3, 8-day composite chlorophyll-a dataset via a THREDDS OPeNDAP service and processes it one time slice at a time. For each 8-day period, the dataset is indexed along the time dimension, and the corresponding calendar date is extracted from the time coordinate. The data are then spatially subset to the Southern Hemisphere mid- to high-latitude region (30–65°S), retaining only the chlorophyll-a variable. Each subset is written to an individual NetCDF file using a date-stamped filename. The workflow includes a file-existence check to skip time steps that have already been processed, enabling safe re-execution without overwriting existing outputs.

The first time you run this it will download every file, but in subsequent years it will only add in new data.

In [ ]:
pp.download_8day(dir_8day)

## Pre-process data

This section constructs seasonal (July–June) Southern Ocean chlorophyll-a datasets from the 8-day composites. Files are grouped by season from the date in each file name (Jul–Dec composites belong to the following year's season), and only complete seasons of 46 composites are processed.

For each season, the files are opened simultaneously using `xarray.open_mfdataset` and concatenated along the time dimension.

Pre-processing follows the approach described in Thomalla et al. (2023). Short gaps in the data (up to two consecutive missing values) are filled using linear interpolation applied sequentially along the longitude, latitude, and time dimensions. This step reduces the impact of transient data gaps while avoiding excessive smoothing of longer missing intervals. A centred three-point rolling mean is subsequently applied along the time dimension to reduce high-frequency noise in the 8-day composites.

Each processed season is written to a single NetCDF file. Existing outputs are skipped, so the workflow is restartable.

In [ ]:
pp.build_seasonal_files(dir_8day, dir_annual, first_season=first_season)

## Generate monthly means

Each seasonal file is resampled to calendar-month means by averaging all 8-day composites falling within each month. Existing outputs are skipped.

In [ ]:
pp.build_monthly_files(dir_annual, dir_monthly)

## Generate climatologies

Annual-mean and monthly-mean climatologies over the climatology seasons, plus the monthly standard error used for the error bars.

Missing values are skipped rather than filled with zero, so pixels with cloud or ice gaps in some years are not biased low.

As in the heat-flux analysis, the standard error is the interannual standard deviation for each calendar month and pixel divided by $\sqrt{N-1}$, where $N$ is the number of climatology seasons (22, so $\sqrt{21}$). It is then area-averaged over each sector.

In [ ]:
pp.build_climatologies(dir_annual, dir_monthly, dir_output, *climatology, variable=variable)

# Produce figures

## Load data and calculate anomalies

In [ ]:
tag = f'{climatology[0]}_{climatology[1]}'

# annual-mean anomaly (map)
xds = xr.open_dataset(f'{dir_annual}/BAMS_chlorophyll_a_southern_ocean_{anomyear}.nc').mean('time')
xds_clim = xr.open_dataset(f'{dir_output}/oc-cci_annual_climatology_{tag}.nc')
xds_anom = xds - xds_clim

# monthly anomaly (annual cycle). Indexed by calendar month so each month is
# compared with its own climatology, then put in Jul-Jun order.
xds_monthly = monthly_in_season_order(f'{dir_monthly}/BAMS_chlorophyll_a_southern_ocean_{anomyear}_monthly.nc')
xds_clim_monthly = xr.open_dataset(f'{dir_output}/oc-cci_monthly_climatology_{tag}.nc')
xds_stderr_monthly = xr.open_dataset(f'{dir_output}/oc-cci_monthly_stderr_{tag}.nc')

xds_anom_monthly = (xds_monthly - xds_clim_monthly).sel(month=SEASON_MONTHS)
xds_stderr_monthly = xds_stderr_monthly.sel(month=SEASON_MONTHS)

## Sector means

Sectors match the heat-flux analysis, all between 40°S and 65°S: Atlantic 70°W–20°E, Indian 20°E–170°E, West Pacific 170°E–135°W, East Pacific 135°W–70°W. Means are area-weighted (exact cell area, equivalent to cos(lat) weighting on this grid).

In [ ]:
area = cell_area(xds_anom_monthly['lat'], xds_anom_monthly['lon'])

anom_means   = {k: v.values for k, v in sector_means(xds_anom_monthly[variable], area).items()}
stderr_means = {k: v.values for k, v in sector_means(xds_stderr_monthly[variable], area).items()}

# print the sector means for the text of the report
for k in anom_means:
    print(f'{k:15s} annual-mean anomaly: {np.nanmean(anom_means[k]):+.3f} mg m-3')

## Fronts (Orsi et al., 1995)

In [ ]:
fronts = [load_orsi_front(f'{dir_fronts}/saf.txt',   nrows=1080),
          load_orsi_front(f'{dir_fronts}/saccf.txt', nrows=912)]

## Map of the annual-mean anomaly

In [ ]:
da_map = xds_anom[variable]
if map_coarsen > 1:
    da_map = da_map.coarsen(lat=map_coarsen, lon=map_coarsen, boundary='trim').mean()

fig, ax = southern_ocean_anomaly_map(da_map, levels=map_levels,
                                     cbar_label='mg m$^{-3}$', fronts=fronts)

os.makedirs(dir_figures, exist_ok=True)
savename = '.'.join(['map-anomaly', str(anomyear), dataset, frequency, resolution, variable, climstr])
fig.savefig(f'{dir_figures}/{savename}.png', dpi=500, bbox_inches='tight')
fig.savefig(f'{dir_figures}/{savename}.pdf', bbox_inches='tight')

## Annual cycle of sector-mean anomalies

In [ ]:
fig, ax = annual_cycle_plot(anom_means, stderr_means,
                            ylabel='Chlorophyll-a anomaly (mg m$^{-3}$)',
                            ylim=(-0.55, 0.55), yticks=np.arange(-0.5, 0.75, 0.25))

savename = '.'.join(['annual-cycle-anomaly', str(anomyear), dataset, frequency, resolution, variable, climstr])
fig.savefig(f'{dir_figures}/{savename}.png', dpi=500, bbox_inches='tight')
fig.savefig(f'{dir_figures}/{savename}.pdf', bbox_inches='tight')